# Cycling Safety Perception Maps: London

Self-contained notebook. Loads data, aggregates, and produces:

**Part A: Interactive HTML maps:**
- GPT safety score map (colour + sigma size)
- Human safety score map (colour + sigma size)
- GPT vs Human disagreement map

**Part B: Static publication maps (PDF, 300 DPI):**
- GPT safety score: 2x2 mosaic (London overview + 3 zoom levels)
- Human safety score: same layout

**Design choices:**
- Transparency: 0.5 for all dots
- Circle size: **large = low sigma (certain), small = high sigma (uncertain)**
- Colour: RdYlGn - Red=unsafe, Green=safe

Run cells in order from top to bottom.

## 1. Install dependencies

In [ ]:
import sys, subprocess
for pkg in ['folium', 'contextily', 'geopandas', 'pyproj',
            'shapely', 'matplotlib-scalebar']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   check=True)
print('Done.')

## 2. Imports

In [ ]:
import os, json, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.patches import PathPatch
from matplotlib.path import Path
import folium
import contextily as ctx
import geopandas as gpd
from shapely.geometry import Point, box
from shapely.ops import unary_union
from scipy.stats import pearsonr, spearmanr
from matplotlib_scalebar.scalebar import ScaleBar
warnings.filterwarnings('ignore')
print('Imports loaded.')

## 3. Configuration

**Key design parameters:**
- `POINT_ALPHA = 0.5` - transparency for all dots (reviewer comment)
- `SIZE_MIN / SIZE_MAX` - **low sigma = large dot, high sigma = small dot** (reviewer comment)
- `ZOOM_LEVELS` - 4 zoom boxes for the 2x2 mosaic static maps

In [ ]:
import numpy as np

# ── PATHS ────────────────────────────────────────────────────────────────────
COORDS_PATH     = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps\metadata_images_mapping_corrected.csv'
GPT_RANK_PATH   = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\gpt_trueskill_rankings_20260621_190450.csv'
HUMAN_RANK_PATH = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data\human_trueskill_rankings_20260621_190450.csv'
OUTPUT_DIR      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Maps'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── COLOUR MAPS ───────────────────────────────────────────────────────────────
CMAP_SCORE    = 'RdYlGn'
CMAP_DISAGREE = 'PuRd'
FOLIUM_SCORE  = 'RdYlGn'

# ── BIN SETTINGS ─────────────────────────────────────────────────────────────
BIN_COLORS  = ['#b2182b', '#e08214', '#1a7837']
BIN_LABELS  = ['Low safety', 'Medium safety', 'High safety']
BIN_MARKERS = ['o', 'D', '*']
BIN_SIZES   = [1.2, 1.2, 1.6]

# ── STATIC MAP SETTINGS ───────────────────────────────────────────────────────
DPI         = 300
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

POINT_ALPHA  = 0.9
SIZE_MIN     = 2
SIZE_MAX     = 12
FOLIUM_R_MIN = 1
FOLIUM_R_MAX = 5

# ── 4-LEVEL ZOOM BOXES ────────────────────────────────────────────────────────
CENTRE_LAT  = 51.5074
CENTRE_LON  = -0.1278

# Holborn centre for the fourth panel
HOLBORN_LAT = 51.5174
HOLBORN_LON = -0.1200

lat           = 51.5
lon_km        = np.cos(np.radians(lat)) * 111.32
lat_km        = 111.32
target_aspect = (18/2 * 0.85) / (14/2 * 0.85)

HALF_WIDTHS_LON = [0.50, 0.20, 0.10, 0.045]
zoom_labels     = ['Greater London', 'Inner London', 'Central London', 'Holborn']

ZOOM_LEVELS = []
for i, (label, half_lon) in enumerate(zip(zoom_labels, HALF_WIDTHS_LON)):
    half_lat   = half_lon * lon_km / (lat_km * target_aspect)
    centre_lat = HOLBORN_LAT if i == 3 else CENTRE_LAT
    centre_lon = HOLBORN_LON if i == 3 else CENTRE_LON
    ZOOM_LEVELS.append({
        'label'  : label,
        'lon_min': centre_lon - half_lon,
        'lon_max': centre_lon + half_lon,
        'lat_min': centre_lat - half_lat,
        'lat_max': centre_lat + half_lat,
    })

print('Configuration loaded.')
print(f'  Output     : {OUTPUT_DIR}')
print(f'  Alpha      : {POINT_ALPHA}')
print(f'  Size range : {SIZE_MIN} to {SIZE_MAX}')
print(f'  Zoom levels: {len(ZOOM_LEVELS)}')
for z in ZOOM_LEVELS:
    print(f"    {z['label']:<16} lon [{z['lon_min']:.3f}, {z['lon_max']:.3f}]  "
          f"lat [{z['lat_min']:.4f}, {z['lat_max']:.4f}]")

In [ ]:
# ── Helper functions for London boundary masking ──────────────────────────────
import io as _io

def geom_to_path(geom):
    """Convert a shapely geometry to a matplotlib Path for boundary masking."""
    def ring_to_verts(ring):
        coords = np.array(ring.coords)
        return coords, [Path.MOVETO] + [Path.LINETO] * (len(coords) - 2) + [Path.CLOSEPOLY]
    verts, codes = [], []
    geoms = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
    for g in geoms:
        v, c = ring_to_verts(g.exterior)
        verts.extend(v); codes.extend(c)
        for interior in g.interiors:
            v, c = ring_to_verts(interior)
            verts.extend(v); codes.extend(c)
    return Path(np.array(verts), codes)


def load_london_boundary():
    """Load and dissolve the London borough boundaries into a single polygon."""
    try:
        url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
                   'master/json/administrative/eng/lad.json')
        resp    = requests.get(url, timeout=20)
        gdf_raw = gpd.read_file(_io.BytesIO(resp.content))
        london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].dissolve().to_crs('EPSG:3857')
        print('London boundary loaded.')
        return london
    except Exception as e:
        print(f'Boundary failed: {e}')
        return None


london_boundary = load_london_boundary()
print('Helper functions defined.')

## 4. Load data

In [ ]:
# ── load corrected coordinates ────────────────────────────────────────────────
coords_df = pd.read_csv(COORDS_PATH)
coords_df['image_name'] = coords_df['image_blob_name'].apply(
    lambda x: os.path.basename(str(x).strip())
)
coords_df['point_id'] = coords_df['image_name'].str.extract(r'^(\d+)_')

point_coords = (
    coords_df.groupby('point_id')[['actual_lat', 'actual_lng']]
    .first().reset_index()
    .rename(columns={'actual_lat': 'lat', 'actual_lng': 'lng'})
)
print(f'Unique locations with corrected coordinates : {len(point_coords):,}')

# ── exclude boat-captured river locations ────────────────────────────────────
# identified by checking points close to the official OS Open Rivers Thames
# geometry, then visually confirming each one against the actual Street View
# photo. 11 locations were confirmed as boat-based captures on the water with
# no cycling relevance. One nearby location (2187) is on a bridge and was kept
# since bridges are legitimate cyclable infrastructure.
RIVER_EXCLUDE_IDS = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '4943', '4354', '1610', '3822', '1984'
]

n_before = len(point_coords)
point_coords = point_coords[
    ~point_coords['point_id'].isin(RIVER_EXCLUDE_IDS)
].reset_index(drop=True)
n_after = len(point_coords)

print(f'Excluded boat-captured river locations      : {n_before - n_after}')
print(f'Locations remaining                          : {n_after:,}')

# ── load rankings ─────────────────────────────────────────────────────────────
gpt_df   = pd.read_csv(GPT_RANK_PATH)
human_df = pd.read_csv(HUMAN_RANK_PATH)
print(f'\nGPT rankings   : {len(gpt_df):,} rows')
print(f'Human rankings : {len(human_df):,} rows')

## 5. Aggregate 4 headings per location

Sigma-weighted mean of TrueSkill mu.
**Note on sigma encoding:** low sigma = more certain = larger dot.

In [ ]:
def normalise_01(series):
    mn, mx = series.min(), series.max()
    if mx > mn:
        return (series - mn) / (mx - mn)
    return pd.Series(np.zeros(len(series)), index=series.index)


def aggregate_location_scores(rank_df, mu_col, sigma_col, label):
    df = rank_df.copy()
    df['image_name'] = df['image_name'].apply(
        lambda x: os.path.basename(str(x).strip())
    )
    df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')
    merged = df.merge(point_coords, on='point_id', how='inner')
    print(f'  {label}: {merged["point_id"].nunique():,} unique locations matched')

    def weighted_agg(grp):
        mu    = grp[mu_col].values
        sigma = grp[sigma_col].values
        w     = 1.0 / np.maximum(sigma ** 2, 1e-12)
        return pd.Series({
            'score_weighted': np.sum(w * mu) / np.sum(w),
            'score_mean'    : mu.mean(),
            'sigma_mean'    : sigma.mean(),
            'n_headings'    : len(grp),
            'lat'           : grp['lat'].iloc[0],
            'lng'           : grp['lng'].iloc[0],
        })

    agg = merged.groupby('point_id').apply(weighted_agg).reset_index()
    agg['score_norm']     = normalise_01(agg['score_weighted'])
    agg['sigma_norm']     = normalise_01(agg['sigma_mean'])
    agg['certainty_norm'] = 1.0 - agg['sigma_norm']
    return agg


def assign_bins(agg_df, score_col='score_weighted'):
    df = agg_df.copy()
    df['bin'] = pd.qcut(
        df[score_col], q=3, labels=[0, 1, 2], duplicates='drop'
    ).astype(int)
    return df


def certainty_to_size(certainty_norm_series):
    return SIZE_MIN + certainty_norm_series * (SIZE_MAX - SIZE_MIN)


def to_gdf(df):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df['lng'], df['lat'])],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')
    return gdf


gpt_agg   = aggregate_location_scores(gpt_df,   'gpt_mu',   'gpt_sigma',   'GPT')
human_agg = aggregate_location_scores(human_df, 'human_mu', 'human_sigma', 'Human')

gpt_plot   = assign_bins(gpt_agg)
human_plot = assign_bins(human_agg)

for df in [gpt_plot, human_plot]:
    df['sigma_norm']     = normalise_01(df['sigma_mean'])
    df['certainty_norm'] = 1.0 - df['sigma_norm']
    df['dot_size']       = certainty_to_size(df['certainty_norm'])

gpt_gdf   = to_gdf(gpt_plot)
human_gdf = to_gdf(human_plot)

disagree_df = gpt_agg[['point_id', 'lat', 'lng', 'score_norm']].merge(
    human_agg[['point_id', 'score_norm']], on='point_id', suffixes=('_gpt', '_human')
)
disagree_df['abs_diff'] = (
    disagree_df['score_norm_gpt'] - disagree_df['score_norm_human']
).abs()
disagree_df['diff_norm'] = normalise_01(disagree_df['abs_diff'])

rho_s, _ = spearmanr(disagree_df['score_norm_gpt'], disagree_df['score_norm_human'])
rho_p, _ = pearsonr(disagree_df['score_norm_gpt'],  disagree_df['score_norm_human'])
print(f'\nGPT vs Human: Spearman rho={rho_s:.4f}  Pearson r={rho_p:.4f}')
print(f'Overlapping locations: {len(disagree_df):,}')

for label, df in [('GPT', gpt_plot), ('Human', human_plot)]:
    counts = df['bin'].value_counts().sort_index()
    print(f'\n{label} bin distribution:')
    for b, lbl in enumerate(BIN_LABELS):
        n = counts.get(b, 0)
        print(f'  Bin {b} ({lbl}): {n:,}  ({n/len(df)*100:.1f}%)')

# Static publication maps (PDF)

**2x2 mosaic layout per source (reviewer comment):**
- Top left: Greater London overview
- Top right: Inner London
- Bottom left: City of London
- Bottom right: Road level (Bank area)

**Encoding:**
- Colour = 3-bin safety class (Red / Yellow / Green)
- Circle size = certainty - **large = low sigma (certain)**
- Alpha = 0.5

Requires internet for basemap tiles. Tiles cached after first run.

In [ ]:
def assign_bins(agg_df, score_col='score_weighted'):
    df = agg_df.copy()
    df['bin'] = pd.qcut(
        df[score_col], q=3, labels=[0, 1, 2], duplicates='drop'
    ).astype(int)
    return df


def certainty_to_size(certainty_norm_series):
    """INVERTED: high certainty (low sigma) = large dot."""
    return SIZE_MIN + certainty_norm_series * (SIZE_MAX - SIZE_MIN)


def to_gdf(df):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df['lng'], df['lat'])],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')
    return gdf


gpt_plot   = assign_bins(gpt_agg)
human_plot = assign_bins(human_agg)

for df in [gpt_plot, human_plot]:
    df['sigma_norm']     = normalise_01(df['sigma_mean'])
    df['certainty_norm'] = 1.0 - df['sigma_norm']
    df['dot_size']       = certainty_to_size(df['certainty_norm'])

gpt_gdf   = to_gdf(gpt_plot)
human_gdf = to_gdf(human_plot)

# updated colours — visible on white basemap
BIN_COLORS  = ['#d73027', '#f4a42c', '#1a9850']
BIN_LABELS  = ['Low safety', 'Medium safety', 'High safety']

for label, df in [('GPT', gpt_plot), ('Human', human_plot)]:
    counts = df['bin'].value_counts().sort_index()
    print(f'{label} bin distribution:')
    for b, lbl in enumerate(BIN_LABELS):
        n = counts.get(b, 0)
        print(f'  Bin {b} ({lbl}): {n:,}  ({n/len(df)*100:.1f}%)')
    print()

In [ ]:
def zoom_to_mercator(zoom):
    corners = gpd.GeoDataFrame(
        geometry=[
            Point(zoom['lon_min'], zoom['lat_min']),
            Point(zoom['lon_max'], zoom['lat_max'])
        ],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')
    return {
        'label' : zoom['label'],
        'x_min' : corners.geometry.iloc[0].x,
        'y_min' : corners.geometry.iloc[0].y,
        'x_max' : corners.geometry.iloc[1].x,
        'y_max' : corners.geometry.iloc[1].y,
    }


zoom_boxes = [zoom_to_mercator(z) for z in ZOOM_LEVELS]

print('Zoom boxes in Web Mercator:')
for z in zoom_boxes:
    print(f'  {z["label"]}')

In [ ]:
import io as _io
from shapely.geometry import box as shapely_box

basemap_per_panel = [
    ctx.providers.CartoDB.Positron,
    ctx.providers.CartoDB.Positron,
    ctx.providers.CartoDB.Positron,
    ctx.providers.CartoDB.PositronNoLabels,
]
size_scale_per_panel = [0.8, 1.5, 3.0, 6.0]

london_boundary = None
print('Loading London boundary...')
try:
    url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
               'master/json/administrative/eng/lad.json')
    resp    = requests.get(url, timeout=20)
    gdf_raw = gpd.read_file(_io.BytesIO(resp.content))
    london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].copy()
    print(f'  London boroughs found: {len(london)}')
    london_boundary = london.dissolve().to_crs('EPSG:3857')
    print(f'  Bounds: {[round(x) for x in london_boundary.total_bounds]}')
except Exception as e:
    print(f'  Failed: {e}')
    from shapely.geometry import MultiPoint
    pts             = MultiPoint(list(zip(gpt_gdf.geometry.x, gpt_gdf.geometry.y)))
    hull            = pts.convex_hull.buffer(3000)
    london_boundary = gpd.GeoDataFrame(geometry=[hull], crs='EPSG:3857')
    print('  Fallback boundary created.')


def add_london_border(ax, boundary):
    if boundary is None:
        return
    try:
        boundary.boundary.plot(
            ax=ax, color='#333333',
            linewidth=1.2, alpha=1.0, zorder=7
        )
    except Exception as e:
        print(f'  Border failed: {e}')


print('Setup complete.')

In [ ]:
from shapely.ops import unary_union

def draw_panel(ax, gdf, title, basemap_source,
               xlim=None, ylim=None, add_scalebar=False,
               size_scale=1.0, show_border=True,
               apply_mask=False):

    ax.set_facecolor('white')
    ax.set_aspect('auto')
    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])

    ctx.add_basemap(
        ax, crs=gdf.crs,
        source=basemap_source, zoom='auto', alpha=0.9
    )

    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])

    if apply_mask and london_boundary is not None:
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()
        buffer     = 5000
        extent     = shapely_box(x0 - buffer, y0 - buffer,
                                 x1 + buffer, y1 + buffer)
        london_u   = unary_union(london_boundary.geometry.values)
        mask_geom  = extent.difference(london_u)
        mask_patch = PathPatch(geom_to_path(mask_geom),
                               fc='white', ec='none', alpha=1.0, zorder=4)
        ax.add_patch(mask_patch)

    for b, (color, marker, msize_factor) in enumerate(
        zip(BIN_COLORS, BIN_MARKERS, BIN_SIZES)
    ):
        subset = gdf[gdf['bin'] == b]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset.geometry.x, subset.geometry.y,
            color=color, marker=marker,
            s=subset['dot_size'] * size_scale * msize_factor,
            alpha=POINT_ALPHA,
            linewidths=0, rasterized=True, zorder=5
        )

    if show_border:
        add_london_border(ax, london_boundary)

    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(ylim[0], ylim[1])

    if add_scalebar:
        ax.add_artist(
            ScaleBar(1, units='m', location='lower right',
                     frameon=True, color='black', box_alpha=0.7,
                     font_properties={'size': 7, 'family': FONT_FAMILY})
        )

    ax.annotate(
        'N', xy=(0.04, 0.12), xycoords='axes fraction',
        fontsize=8, fontfamily=FONT_FAMILY, fontweight='bold',
        ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='grey', alpha=0.8),
        zorder=8
    )
    ax.annotate(
        '', xy=(0.04, 0.12), xycoords='axes fraction',
        xytext=(0.04, 0.06), textcoords='axes fraction',
        arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
        zorder=8
    )

    ax.set_title(title, fontsize=13, fontfamily=FONT_FAMILY,
                 fontweight='bold', pad=6)

    ax.tick_params(left=False, bottom=False,
                   labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)


print('draw_panel defined.')

In [ ]:
print('Building GPT 2x2 mosaic map...')

gpt_gdf_plot = gpt_gdf.copy()

fig_gpt, axes_gpt = plt.subplots(2, 2, figsize=(18, 14),
                                  gridspec_kw={'wspace': 0, 'hspace': 0.05})
fig_gpt.patch.set_facecolor('white')
fig_gpt.suptitle(
    f'GPT Cycling Safety Score — London  (n = {len(gpt_gdf_plot):,} locations)\n'
    'Colour and shape: 3-bin quantile class  |  \u03b1 = 0.8',
    fontsize=13, fontfamily=FONT_FAMILY, fontweight='bold', y=1.01
)

panel_positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for idx, ((row, col), zbox) in enumerate(zip(panel_positions, zoom_boxes)):
    ax = axes_gpt[row, col]
    draw_panel(
        ax, gpt_gdf_plot, zbox['label'],
        basemap_source=basemap_per_panel[idx],
        xlim=(zbox['x_min'], zbox['x_max']),
        ylim=(zbox['y_min'], zbox['y_max']),
        add_scalebar=True,
        size_scale=size_scale_per_panel[idx],
        show_border=True,
        apply_mask=(idx == 0)
    )

shape_handles = [
    mlines.Line2D([0], [0], marker='o', color='w',
                  markerfacecolor='#b2182b', markersize=10, label='Low safety'),
    mlines.Line2D([0], [0], marker='D', color='w',
                  markerfacecolor='#e08214', markersize=10, label='Medium safety'),
    mlines.Line2D([0], [0], marker='*', color='w',
                  markerfacecolor='#1a7837', markersize=14, label='High safety'),
]

fig_gpt.legend(
    handles=shape_handles,
    loc='lower center', ncol=3, fontsize=12,
    frameon=True, framealpha=0.95,
    prop={'family': FONT_FAMILY, 'size': 12},
    bbox_to_anchor=(0.5, 0.01),
    title='Safety class (colour and shape)',
    title_fontproperties={
        'family': FONT_FAMILY, 'size': 12, 'weight': 'bold'
    },
    edgecolor='#cccccc'
)

fig_gpt.tight_layout(pad=0.5)
out_gpt = os.path.join(OUTPUT_DIR, 'static_gpt_score_mosaic.pdf')
fig_gpt.savefig(out_gpt, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_gpt}')

## checking problems visually

In [ ]:
import io
from PIL import Image
from azure.storage.blob import BlobServiceClient

AZURE_CONN_STR = (
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
BLOB_CONTAINER = 'london'
IMAGE_PREFIX   = 'training/training/Images/'

blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

def fetch_image(image_name):
    blob_data = container_client.download_blob(
        IMAGE_PREFIX + image_name, timeout=30
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')

# all 12 candidates checked, sorted by distance to the Thames
candidate_ids = [
    '2705', '2138', '4577', '395', '2028',
    '2858', '2187', '4943', '4354', '1610', '3822', '1984'
]
decisions = {
    '2705': 'Excluded — boat on river',
    '2138': 'Excluded — boat on river',
    '4577': 'Excluded — boat on river',
    '395' : 'Excluded — boat on river',
    '2028': 'Excluded — boat on river',
    '2858': 'Excluded — boat on river',
    '2187': 'Kept — on bridge',
    '4943': 'Excluded — boat on river',
    '4354': 'Excluded — boat on river',
    '1610': 'Excluded — boat on river',
    '3822': 'Excluded — boat on river',
    '1984': 'Excluded — boat on river',
}

fig, axes = plt.subplots(len(candidate_ids), 4, figsize=(16, 4 * len(candidate_ids)))
fig.patch.set_facecolor('white')

for row, pid in enumerate(candidate_ids):
    decision = decisions[pid]
    colour   = '#c0392b' if 'Excluded' in decision else '#1a9850'

    for col, heading in enumerate([0, 90, 180, 270]):
        ax = axes[row, col]
        ax.set_facecolor('white')
        img_name = f'{pid}_{heading}.jpg'
        try:
            img = fetch_image(img_name)
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'Not found', ha='center', va='center',
                     transform=ax.transAxes, fontsize=8, color='red')

        if col == 0:
            ax.set_ylabel(f'Point {pid}\n{decision}', fontsize=9,
                           color=colour, fontweight='bold')
        ax.set_title(f'{heading}\u00b0', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'river_exclusion_verification.pdf')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

# Human

In [ ]:
print('Building Human 2x2 mosaic map...')

human_gdf_plot = human_gdf.copy()

fig_human, axes_human = plt.subplots(2, 2, figsize=(18, 14),
                                      gridspec_kw={'wspace': 0, 'hspace': 0.05})
fig_human.patch.set_facecolor('white')
fig_human.suptitle(
    f'Human Cycling Safety Score — London  (n = {len(human_gdf_plot):,} locations)\n'
    'Colour and shape: 3-bin quantile class  |  \u03b1 = 0.8',
    fontsize=13, fontfamily=FONT_FAMILY, fontweight='bold', y=1.01
)

for idx, ((row, col), zbox) in enumerate(zip(panel_positions, zoom_boxes)):
    ax = axes_human[row, col]
    draw_panel(
        ax, human_gdf_plot, zbox['label'],
        basemap_source=basemap_per_panel[idx],
        xlim=(zbox['x_min'], zbox['x_max']),
        ylim=(zbox['y_min'], zbox['y_max']),
        add_scalebar=True,
        size_scale=size_scale_per_panel[idx],
        show_border=True,
        apply_mask=(idx == 0)
    )

fig_human.legend(
    handles=shape_handles,
    loc='lower center', ncol=3, fontsize=12,
    frameon=True, framealpha=0.95,
    prop={'family': FONT_FAMILY, 'size': 12},
    bbox_to_anchor=(0.5, 0.01),
    title='Safety class (colour and shape)',
    title_fontproperties={
        'family': FONT_FAMILY, 'size': 12, 'weight': 'bold'
    },
    edgecolor='#cccccc'
)

fig_human.tight_layout(pad=0.5)
out_human = os.path.join(OUTPUT_DIR, 'static_human_score_mosaic.pdf')
fig_human.savefig(out_human, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_human}')

In [ ]:
print('=' * 60)
print('ALL FILES SAVED')
print('=' * 60)
print(f'  Output folder: {OUTPUT_DIR}')
print()
print('  HTML maps (open in browser):')
print('    map_gpt_score.html')
print('    map_human_score.html')
print('    map_disagreement.html')
print()
print('  Static PDFs (for paper):')
print('    static_gpt_score_mosaic.pdf   (2x2: Greater London / Inner / City / Road level)')
print('    static_human_score_mosaic.pdf (same layout)')
print()
print(f'  GPT locations   : {len(gpt_agg):,}')
print(f'  Human locations : {len(human_agg):,}')
print(f'  Spearman rho    : {rho_s:.4f}')
print(f'  Pearson r       : {rho_p:.4f}')

# Extra things for the paper

In [ ]:
import io
import time
import requests as req
from PIL import Image
from azure.storage.blob import BlobServiceClient

AZURE_CONN_STR = (
    'DefaultEndpointsProtocol=https;AccountName=YOUR_STORAGE_ACCOUNT;'
    'AccountKey=YOUR_AZURE_STORAGE_ACCOUNT_KEY;'
    'EndpointSuffix=core.windows.net'
)
blob_svc         = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client('london')


def fetch_image(image_name):
    blob_data = container_client.download_blob(
        'training/training/Images/' + image_name, timeout=30
    ).readall()
    return Image.open(io.BytesIO(blob_data)).convert('RGB')


def reverse_geocode(lat, lng):
    try:
        url    = 'https://nominatim.openstreetmap.org/reverse'
        params = {
            'lat': lat, 'lon': lng,
            'format': 'json', 'zoom': 17,
            'addressdetails': 1
        }
        headers = {'User-Agent': 'cycling-safety-research/1.0'}
        r    = req.get(url, params=params, headers=headers, timeout=10)
        data = r.json()
        addr = data.get('address', {})
        road = addr.get('road', '')
        area = (addr.get('suburb') or addr.get('neighbourhood')
                or addr.get('quarter') or addr.get('city_district', ''))
        if road and area:   return f'{road}, {area}'
        elif road:          return road
        elif area:          return area
        else:               return data.get('display_name', '').split(',')[0]
    except Exception:
        return 'Location unknown'


def get_example_images(rank_df, mu_col, sigma_col,
                        agg_df, n_top=2, n_mid=2, n_bottom=2,
                        min_headings=2, exclude_ids=None):
    df = rank_df.copy()
    df['image_name'] = df['image_name'].apply(
        lambda x: os.path.basename(str(x).strip())
    )
    df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')

    sorted_agg = agg_df.sort_values(
        'score_weighted', ascending=False
    ).reset_index(drop=True)

    sorted_agg = sorted_agg[sorted_agg['n_headings'] >= min_headings]
    if exclude_ids:
        sorted_agg = sorted_agg[~sorted_agg['point_id'].isin(
            [str(x) for x in exclude_ids]
        )]

    n_total    = len(sorted_agg)
    top_ids    = sorted_agg.head(n_top)['point_id'].tolist()
    bottom_ids = sorted_agg.tail(n_bottom).sort_values(
        'score_weighted'
    )['point_id'].tolist()
    mid_start  = n_total // 2 - n_mid // 2
    mid_ids    = sorted_agg.iloc[mid_start:mid_start + n_mid]['point_id'].tolist()

    print(f'  Top {n_top} (safest)      : {top_ids}')
    print(f'  Mid {n_mid} (medium)      : {mid_ids}')
    print(f'  Bottom {n_bottom} (dangerous) : {bottom_ids}')

    results = []
    for pid, category, color in (
        [(p, 'High safety',   '#1a9850') for p in top_ids]    +
        [(p, 'Medium safety', '#f4a42c') for p in mid_ids]    +
        [(p, 'Low safety',    '#d73027') for p in bottom_ids]
    ):
        grp = df[df['point_id'] == pid]
        if len(grp) == 0:
            continue
        best     = grp.loc[grp[sigma_col].idxmin()]
        loc_info = agg_df[agg_df['point_id'] == pid].iloc[0]
        lat = loc_info['lat']
        lng = loc_info['lng']
        print(f'    Geocoding point {pid} ({lat:.4f}, {lng:.4f})...')
        location_name = reverse_geocode(lat, lng)
        time.sleep(1.1)
        results.append({
            'point_id'     : pid,
            'image_name'   : best['image_name'],
            'mu'           : loc_info['score_weighted'],
            'sigma'        : best[sigma_col],
            'score_norm'   : loc_info['score_norm'],
            'category'     : category,
            'color'        : color,
            'location_name': location_name,
            'lat'          : lat,
            'lng'          : lng,
        })
    return results


def plot_streetview_examples(rank_df, mu_col, sigma_col,
                              agg_df, source_label, output_path,
                              n_top=2, n_mid=2, n_bottom=2,
                              min_headings=2, exclude_ids=None):

    print(f'\n{source_label} — selecting locations...')
    examples = get_example_images(
        rank_df, mu_col, sigma_col, agg_df,
        n_top, n_mid, n_bottom, min_headings, exclude_ids
    )
    n_cols = len(examples)

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5))
    fig.patch.set_facecolor('white')
    fig.suptitle(
        f'{source_label}',
        fontsize=11, fontweight='bold', fontfamily=FONT_FAMILY, y=1.04
    )

    for col_idx, (ex, ax) in enumerate(zip(examples, axes)):
        ax.set_facecolor('white')
        try:
            pil_img = fetch_image(ex['image_name'])
            ax.imshow(pil_img)
        except Exception as e:
            ax.text(0.5, 0.5, f'Could not load\n{str(e)[:50]}',
                    ha='center', va='center',
                    transform=ax.transAxes, fontsize=7, color='red')
            ax.set_facecolor('#f8f8f8')

        for spine in ax.spines.values():
            spine.set_edgecolor(ex['color'])
            spine.set_linewidth(4)

        ax.text(0.5, 1.01, ex['category'],
                transform=ax.transAxes,
                ha='center', va='bottom',
                fontsize=9, fontweight='bold',
                color=ex['color'],
                fontfamily=FONT_FAMILY)

        heading = ex['image_name'].replace('.jpg', '').split('_')[-1]
        caption = (
            f'{ex["location_name"]}\n'
            f'Heading {heading}\u00b0  |  '
            f'TrueSkill \u03bc = {ex["mu"]:.3f}  |  '
            f'\u03c3 = {ex["sigma"]:.3f}'
        )
        ax.set_xlabel(
            caption, fontsize=7.5,
            fontfamily=FONT_FAMILY,
            labelpad=8, color='#333333'
        )
        ax.set_xticks([])
        ax.set_yticks([])

    for div in [n_top / n_cols, (n_top + n_mid) / n_cols]:
        fig.add_artist(
            plt.Line2D(
                [div, div], [0.08, 0.95],
                transform=fig.transFigure,
                color='#aaaaaa', linewidth=1.5, linestyle='--'
            )
        )

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved: {output_path}')


# ── exclude bad locations ─────────────────────────────────────────────────────
GPT_EXCLUDE   = ['4140', '2245']
HUMAN_EXCLUDE = ['2028', '4577']

# ── run for GPT ───────────────────────────────────────────────────────────────
print('Building GPT street view examples...')
plot_streetview_examples(
    rank_df      = gpt_df,
    mu_col       = 'gpt_mu',
    sigma_col    = 'gpt_sigma',
    agg_df       = gpt_agg,
    source_label = 'Panel C - GPT-5.2 synthetic votes',
    output_path  = os.path.join(OUTPUT_DIR, 'streetview_examples_gpt.pdf'),
    n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=GPT_EXCLUDE
)

# ── run for Human ─────────────────────────────────────────────────────────────
print('Building Human street view examples...')
plot_streetview_examples(
    rank_df      = human_df,
    mu_col       = 'human_mu',
    sigma_col    = 'human_sigma',
    agg_df       = human_agg,
    source_label = 'Panel B - Human crowdsourced votes',
    output_path  = os.path.join(OUTPUT_DIR, 'streetview_examples_human.pdf'),
    n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=HUMAN_EXCLUDE
)

# ── store examples as variables for combined figure ───────────────────────────
print('\nStoring examples for combined figure...')
examples_human = get_example_images(
    rank_df=human_df, mu_col='human_mu', sigma_col='human_sigma',
    agg_df=human_agg, n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=HUMAN_EXCLUDE
)
examples_gpt = get_example_images(
    rank_df=gpt_df, mu_col='gpt_mu', sigma_col='gpt_sigma',
    agg_df=gpt_agg, n_top=2, n_mid=2, n_bottom=2,
    exclude_ids=GPT_EXCLUDE
)
print(f'\nexamples_human : {len(examples_human)} locations ready')
print(f'examples_gpt   : {len(examples_gpt)} locations ready')

## Same chart as the previous one but merged

In [ ]:
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import PathPatch
from matplotlib.path import Path
from shapely.geometry import Point, box
from shapely.ops import unary_union
import numpy as np
import requests as req
import io
import os

SAFETY_COLORS = {
    'High safety'  : '#1a9850',
    'Medium safety': '#f4a42c',
    'Low safety'   : '#d73027',
}
FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
DPI = 300
MAX_NAME_LEN = 30


def truncate(name, maxlen=MAX_NAME_LEN):
    return name if len(name) <= maxlen else name[:maxlen - 1] + '\u2026'


def assign_labels(examples, prefix):
    labelled = []
    counters = {'High safety': 1, 'Medium safety': 3, 'Low safety': 5}
    used     = {'High safety': 0, 'Medium safety': 0, 'Low safety': 0}
    for ex in examples:
        cat   = ex['category']
        idx   = counters[cat] + used[cat]
        label = f'{prefix}{idx}'
        used[cat] += 1
        labelled.append({**ex, 'label': label})
    return labelled


def geom_to_path(geom):
    def ring_to_verts(ring):
        coords = np.array(ring.coords)
        return coords, [Path.MOVETO] + [Path.LINETO] * (len(coords) - 2) + [Path.CLOSEPOLY]
    verts, codes = [], []
    geoms = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
    for g in geoms:
        v, c = ring_to_verts(g.exterior)
        verts.extend(v); codes.extend(c)
        for interior in g.interiors:
            v, c = ring_to_verts(interior)
            verts.extend(v); codes.extend(c)
    return Path(np.array(verts), codes)


def load_london_boundary():
    try:
        url     = ('https://raw.githubusercontent.com/martinjc/UK-GeoJSON/'
                   'master/json/administrative/eng/lad.json')
        resp    = req.get(url, timeout=20)
        gdf_raw = gpd.read_file(io.BytesIO(resp.content))
        london  = gdf_raw[gdf_raw['LAD13CD'].str.startswith('E09')].dissolve().to_crs('EPSG:3857')
        print('London boundary loaded.')
        return london
    except Exception as e:
        print(f'Boundary failed: {e}')
        return None


def draw_panel(fig, gs_panel, examples, prefix,
               panel_label, london_boundary):
    examples_l = assign_labels(examples, prefix)
    rows = {
        'High safety'  : [ex for ex in examples_l if ex['category'] == 'High safety'],
        'Medium safety': [ex for ex in examples_l if ex['category'] == 'Medium safety'],
        'Low safety'   : [ex for ex in examples_l if ex['category'] == 'Low safety'],
    }

    locs_df = gpd.GeoDataFrame(
        [{'lat': ex['lat'], 'lng': ex['lng'],
          'category': ex['category'], 'label': ex['label']}
         for ex in examples_l],
        geometry=[Point(ex['lng'], ex['lat']) for ex in examples_l],
        crs='EPSG:4326'
    ).to_crs('EPSG:3857')

    # ── map ───────────────────────────────────────────────────────────────────
    ax_map = fig.add_subplot(gs_panel[:, 0])
    ax_map.set_facecolor('white')
    ax_map.set_aspect('auto')
    ax_map.set_xlim(-57886, 42000)
    ax_map.set_ylim(6668000, 6746062)

    ctx.add_basemap(ax_map, crs='EPSG:3857',
                    source=ctx.providers.CartoDB.Positron,
                    zoom='auto', alpha=0.9)
    ax_map.set_xlim(-57886, 42000)
    ax_map.set_ylim(6668000, 6746062)

    if london_boundary is not None:
        extent     = box(-57886, 6668000, 42000, 6746062)
        london_u   = unary_union(london_boundary.geometry.values)
        mask_geom  = extent.difference(london_u)
        mask_patch = PathPatch(geom_to_path(mask_geom),
                               fc='white', ec='none', alpha=1.0, zorder=4)
        ax_map.add_patch(mask_patch)
        london_boundary.boundary.plot(
            ax=ax_map, color='#333333', linewidth=1.2, zorder=5
        )

    for _, row in locs_df.iterrows():
        ax_map.scatter(
            row.geometry.x, row.geometry.y,
            color=SAFETY_COLORS[row['category']],
            marker='o', s=180,
            zorder=6, edgecolors='white', linewidths=1.2
        )
        ax_map.annotate(
            row['label'],
            xy=(row.geometry.x, row.geometry.y),
            xytext=(7, 7), textcoords='offset points',
            fontsize=7.5, fontweight='bold',
            color='#222222', zorder=7
        )

    ax_map.set_axis_off()

    # ── image grid ────────────────────────────────────────────────────────────
    safety_classes = ['High safety', 'Medium safety', 'Low safety']
    all_img_axes   = []

    for row_idx, cat in enumerate(safety_classes):
        cat_examples = rows[cat]
        color        = SAFETY_COLORS[cat]

        for col_idx, ex in enumerate(cat_examples[:2]):
            ax = fig.add_subplot(gs_panel[row_idx, col_idx + 1])
            ax.set_facecolor('white')
            all_img_axes.append(ax)

            try:
                pil_img = fetch_image(ex['image_name'])
                ax.imshow(pil_img)
            except Exception as e:
                ax.text(0.5, 0.5, f'Could not load\n{str(e)[:40]}',
                        ha='center', va='center',
                        transform=ax.transAxes, fontsize=6, color='red')

            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3.5)

            # label and street name in caption only — no title above image
            caption = (f'{ex["label"]} — {truncate(ex["location_name"])}'
                       f'\n\u03bc = {ex["mu"]:.1f}')
            ax.set_xlabel(caption, fontsize=8, labelpad=5, color='#333333')
            ax.set_xticks([])
            ax.set_yticks([])

        # row safety class label on right
        ax_lbl = fig.add_subplot(gs_panel[row_idx, 3])
        ax_lbl.set_axis_off()
        ax_lbl.text(0.1, 0.5, cat,
                    transform=ax_lbl.transAxes,
                    fontsize=8.5, fontweight='bold',
                    color=color, va='center', ha='left', rotation=270)

    # single panel label centred above the full panel (map + images)
    fig.canvas.draw()
    map_pos       = ax_map.get_position()
    last_img_pos  = all_img_axes[1].get_position()
    panel_centre_x = (map_pos.x0 + last_img_pos.x1) / 2
    panel_top_y    = max(map_pos.y1, all_img_axes[0].get_position().y1)

    fig.text(
        panel_centre_x,
        panel_top_y + 0.008,
        panel_label,
        fontsize=10, fontweight='bold',
        ha='center', va='bottom'
    )


# ── load boundary ─────────────────────────────────────────────────────────────
london_boundary = load_london_boundary()

# ── figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('white')

outer_gs = gridspec.GridSpec(
    3, 1, figure=fig,
    height_ratios=[1, 1, 0.04],
    hspace=0.06
)

gs_human = gridspec.GridSpecFromSubplotSpec(
    3, 4, subplot_spec=outer_gs[0],
    width_ratios=[3.5, 1, 1, 0.18],
    height_ratios=[1, 1, 1],
    hspace=0.22, wspace=0.02
)
gs_gpt = gridspec.GridSpecFromSubplotSpec(
    3, 4, subplot_spec=outer_gs[1],
    width_ratios=[3.5, 1, 1, 0.18],
    height_ratios=[1, 1, 1],
    hspace=0.22, wspace=0.02
)

draw_panel(fig, gs_human, examples_human, 'H',
           'Panel A — Human crowdsourced votes', london_boundary)
draw_panel(fig, gs_gpt,   examples_gpt,   'G',
           'Panel B — GPT-5.2 synthetic votes',  london_boundary)

# ── shared legend ─────────────────────────────────────────────────────────────
ax_legend = fig.add_subplot(outer_gs[2])
ax_legend.set_axis_off()
legend_elements = [
    mpatches.Patch(color='#1a9850', label='High safety'),
    mpatches.Patch(color='#f4a42c', label='Medium safety'),
    mpatches.Patch(color='#d73027', label='Low safety'),
]
ax_legend.legend(
    handles=legend_elements,
    loc='center', ncol=3,
    fontsize=10, frameon=True,
    framealpha=0.95, edgecolor='#cccccc'
)

fig.suptitle(
    'Perceived cycling safety scores across Greater London',
    fontsize=13, fontweight='bold', y=1.002
)

out_path = os.path.join(OUTPUT_DIR, 'figure6_streetview_combined.pdf')
fig.savefig(out_path, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')